In [1]:
import os, glob
import sys
import json
from PIL import Image
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import tifffile as tiff
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import cv2

import pandas as pd

from sklearn.model_selection import KFold

sys.path.append("/kaggle/input/detection-wheel")

In [2]:
import torch
torch.__version__  #torch 2.0
#!nvidia-smi   #CUDA Version: 11.4
! ls /usr/local

bin   cuda-11	 cuda-12    etc    include  man     sbin   src
cuda  cuda-11.8  cuda-12.1  games  lib	    nvidia  share


In [3]:
# # Install pycocotools package
import os
!mkdir /kaggle/working/packages
!cp -r /kaggle/input/pycocotools/* /kaggle/working/packages
os.chdir("/kaggle/working/packages/pycocotools-2.0.6/")
!python setup.py install -q
!pip install . --no-index --find-links /kaggle/working/packages/ -q
# # Install mmcv and mmdet packages
# #3.0
# #!pip install mmcv mmdet --no-index --find-links /kaggle/input/mmdetection/ -q
# #os.chdir("/kaggle/working")
# #ytt 2140
# !pip install '/kaggle/input/mmdetectionv2140/addict-2.4.0-py3-none-any.whl' --no-deps
# !pip install '/kaggle/input/mmdetectionv2140/yapf-0.31.0-py2.py3-none-any.whl' --no-deps
# !pip install '/kaggle/input/mmdetectionv2140/terminal-0.4.0-py3-none-any.whl' --no-deps
# !pip install '/kaggle/input/mmdetectionv2140/terminaltables-3.1.0-py3-none-any.whl' --no-deps
# !pip install '/kaggle/input/mmdetectionv2140/mmcv_full-1_3_8-cu110-torch1_7_0/mmcv_full-1.3.8-cp37-cp37m-manylinux1_x86_64.whl' --no-deps
# !pip install '/kaggle/input/mmdetectionv2140/pycocotools-2.0.2/pycocotools-2.0.2' --no-deps
# !pip install '/kaggle/input/mmdetectionv2140/mmpycocotools-12.0.3/mmpycocotools-12.0.3' --no-deps

# !rm -rf mmdetection

# !cp -r /kaggle/input/mmdetectionv2140/mmdetection-2.14.0 /kaggle/working/
# !mv /kaggle/working/mmdetection-2.14.0 /kaggle/working/mmdetection
# %cd /kaggle/working/mmdetection
# !pip install -e .


# 必要なライブラリのインストール（オフライン用）
!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/addict-2.4.0-py3-none-any.whl
!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/yapf-0.32.0-py2.py3-none-any.whl
!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/terminal-0.4.0-py3-none-any.whl
!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/terminaltables-3.1.10-py2.py3-none-any.whl
#ytt
#!pip install /kaggle/input/2023-hhp-mmdet/mmcv_full-1.7.0-cp310-cp310-manylinux1_x86_64_cu113.whl
!pip install /kaggle/input/mmdet3-wheels/mmcv_full-1.7.1-cp310-cp310-linux_x86_64.whl
#!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/mmcv_full-1.7.0-cp37-cp37m-linux_x86_64.whl
#!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/pycocotools-2.0.6-cp37-cp37m-linux_x86_64.whl
#!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/mmpycocotools-12.0.3-cp37-cp37m-linux_x86_64.whl
!cp -r /kaggle/input/cbnetv2-repo/cbnet_repo /kaggle/working/
# !cp -r /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/mmdetection/ /kaggle/working/
%cd /kaggle/working/cbnet_repo
!pip install -e . --no-deps
%cd /kaggle/working/

!pip install /kaggle/input/mmdetection-2-26-0/mmdetection-2-26-0/mmdet-2.26.0-py3-none-any.whl


running install
/opt/conda/lib/python3.10/site-packages/setuptools/command/install.py:34: SetuptoolsDeprecationWarning: setup.py install is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/setuptools/command/easy_install.py:156: EasyInstallDeprecationWarning: easy_install command is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(
running bdist_egg
running egg_info
writing pycocotools.egg-info/PKG-INFO
writing dependency_links to pycocotools.egg-info/dependency_links.txt
writing requirements to pycocotools.egg-info/requires.txt
writing top-level names to pycocotools.egg-info/top_level.txt
reading manifest file 'pycocotools.egg-info/SOURCES.txt'
reading manifest template 'MANIFEST.in'
writing manifest file 'pycocotools.egg-info/SOURCES.txt'
installing library code to build/bdist.linux-x86_64/egg
running install_lib
running build_py
creating build
creating build/lib.linux-x86_64-3.10
cre

In [4]:
!pip install /kaggle/input/enseibleboxes109/ensemble_boxes-1.0.9-py3-none-any.whl 

Processing /kaggle/input/enseibleboxes109/ensemble_boxes-1.0.9-py3-none-any.whl


In [5]:
import base64
import numpy as np
from pycocotools import _mask as coco_mask
import typing as t
import zlib

def encode_binary_mask(mask: np.ndarray) -> t.Text:
  """Converts a binary mask into OID challenge encoding ascii text."""

  # check input mask --
  if mask.dtype != np.bool:
    raise ValueError(
        "encode_binary_mask expects a binary mask, received dtype == %s" %
        mask.dtype)

  mask = np.squeeze(mask)
  if len(mask.shape) != 2:
    raise ValueError(
        "encode_binary_mask expects a 2d mask, received shape == %s" %
        mask.shape)

  # convert input mask to expected COCO API input --
  mask_to_encode = mask.reshape(mask.shape[0], mask.shape[1], 1)
  mask_to_encode = mask_to_encode.astype(np.uint8)
  mask_to_encode = np.asfortranarray(mask_to_encode)

  # RLE encode mask --
  encoded_mask = coco_mask.encode(mask_to_encode)[0]["counts"]

  # compress and base64 encoding --
  binary_str = zlib.compress(encoded_mask, zlib.Z_BEST_COMPRESSION)
  base64_str = base64.b64encode(binary_str)
  return base64_str

In [6]:
import os
import numpy as np
import torch
from PIL import Image


class PennFudanDataset(torch.utils.data.Dataset):
    def __init__(self, imgs, transforms):
        self.transforms = transforms
        # load all image files, sorting them to
        # ensure that they are aligned
        self.imgs = imgs
        self.name_indices = [os.path.splitext(os.path.basename(i))[0] for i in imgs]

    def __getitem__(self, idx):
        # load images and masks
        img_path = self.imgs[idx]
        name = self.name_indices[idx]
        array = tiff.imread(img_path)
        img = Image.fromarray(array)
        
        img, _ = self.transforms(img, img)

        return img, name

    def __len__(self):
        return len(self.imgs)

In [7]:
%%writefile wbf_tracking.py

# coding: utf-8

__author__ = 'ZFTurbo: https://kaggle.com/zfturbo'
# Modified by Mista G: https://www.kaggle.com/mistag

import warnings
import numpy as np
from numba import jit

@jit(nopython=True)
def bb_intersection_over_union(A, B) -> float:
    xA = max(A[0], B[0])
    yA = max(A[1], B[1])
    xB = min(A[2], B[2])
    yB = min(A[3], B[3])

    # compute the area of intersection rectangle
    interArea = max(0, xB - xA) * max(0, yB - yA)

    if interArea == 0:
        return 0.0

    # compute the area of both the prediction and ground-truth rectangles
    boxAArea = (A[2] - A[0]) * (A[3] - A[1])
    boxBArea = (B[2] - B[0]) * (B[3] - B[1])

    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou


def prefilter_boxes(boxes, scores, labels, weights, thr):
    # Create dict with boxes stored by its label
    new_boxes = dict()

    for t in range(len(boxes)):

        if len(boxes[t]) != len(scores[t]):
            print('Error. Length of boxes arrays not equal to length of scores array: {} != {}'.format(len(boxes[t]), len(scores[t])))
            sys.exit()

        if len(boxes[t]) != len(labels[t]):
            print('Error. Length of boxes arrays not equal to length of labels array: {} != {}'.format(len(boxes[t]), len(labels[t])))
            sys.exit()

        for j in range(len(boxes[t])):
            score = scores[t][j]
            if score < thr:
                continue
            label = int(labels[t][j])
            box_part = boxes[t][j]
            x1 = max(float(box_part[0]), 0.)
            y1 = max(float(box_part[1]), 0.)
            x2 = max(float(box_part[2]), 0.)
            y2 = max(float(box_part[3]), 0.)

            # Box data checks
            if x2 < x1:
                warnings.warn('X2 < X1 value in box. Swap them.')
                x1, x2 = x2, x1
            if y2 < y1:
                warnings.warn('Y2 < Y1 value in box. Swap them.')
                y1, y2 = y2, y1
            if x1 > 1:
                warnings.warn('X1 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
                x1 = 1
            if x2 > 1:
                warnings.warn('X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
                x2 = 1
            if y1 > 1:
                warnings.warn('Y1 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
                y1 = 1
            if y2 > 1:
                warnings.warn('Y2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
                y2 = 1
            if (x2 - x1) * (y2 - y1) == 0.0:
                warnings.warn("Zero area box skipped: {}.".format(box_part))
                continue

            # [label, score, weight, model index, x1, y1, x2, y2]
            b = [int(label), float(score) * weights[t], weights[t], t, x1, y1, x2, y2]
            if label not in new_boxes:
                new_boxes[label] = []
            new_boxes[label].append(b)

    # Sort each list in dict by score and transform it to numpy array
    for k in new_boxes:
        current_boxes = np.array(new_boxes[k])
        new_boxes[k] = current_boxes[current_boxes[:, 1].argsort()[::-1]]

    return new_boxes


def get_weighted_box(boxes, conf_type='avg'):
    """
    Create weighted box for set of boxes
    :param boxes: set of boxes to fuse
    :param conf_type: type of confidence one of 'avg' or 'max'
    :return: weighted box (label, score, weight, x1, y1, x2, y2)
    """

    box = np.zeros(8, dtype=np.float32)
    conf = 0
    conf_list = []
    w = 0
    for b in boxes:
        box[4:] += (b[1] * b[4:])
        conf += b[1]
        conf_list.append(b[1])
        w += b[2]
    box[0] = boxes[0][0]
    if conf_type == 'avg':
        box[1] = conf / len(boxes)
    elif conf_type == 'max':
        box[1] = np.array(conf_list).max()
    elif conf_type in ['box_and_model_avg', 'absent_model_aware_avg']:
        box[1] = conf / len(boxes)
    box[2] = w
    box[3] = -1 # model index field is retained for consistensy but is not used.
    box[4:] /= conf
    return box


def find_matching_box(boxes_list, new_box, match_iou):
    best_iou = match_iou
    best_index = -1
    for i in range(len(boxes_list)):
        box = boxes_list[i]
        if box[0] != new_box[0]:
            continue
        iou = bb_intersection_over_union(box[4:], new_box[4:])
        if iou > best_iou:
            best_index = i
            best_iou = iou

    return best_index, best_iou


def weighted_boxes_fusion_tracking(boxes_list, scores_list, labels_list, weights=None, iou_thr=0.55, skip_box_thr=0.0, conf_type='avg', allows_overflow=False):
    '''
    :param boxes_list: list of boxes predictions from each model, each box is 4 numbers.
    It has 3 dimensions (models_number, model_preds, 4)
    Order of boxes: x1, y1, x2, y2. We expect float normalized coordinates [0; 1]
    :param scores_list: list of scores for each model
    :param labels_list: list of labels for each model
    :param weights: list of weights for each model. Default: None, which means weight == 1 for each model
    :param iou_thr: IoU value for boxes to be a match
    :param skip_box_thr: exclude boxes with score lower than this variable
    :param conf_type: how to calculate confidence in weighted boxes. 'avg': average value, 'max': maximum value, 'box_and_model_avg': box and model wise hybrid weighted average, 'absent_model_aware_avg': weighted average that takes into account the absent model.
    :param allows_overflow: false if we want confidence score not exceed 1.0

    :return: boxes: boxes coordinates (Order of boxes: x1, y1, x2, y2).
    :return: scores: confidence scores
    :return: labels: boxes labels
    :return: wbfo: original boxes coordinates for each fused box
    '''

    if weights is None:
        weights = np.ones(len(boxes_list))
    if len(weights) != len(boxes_list):
        print('Warning: incorrect number of weights {}. Must be: {}. Set weights equal to 1.'.format(len(weights), len(boxes_list)))
        weights = np.ones(len(boxes_list))
    weights = np.array(weights)

    if conf_type not in ['avg', 'max', 'box_and_model_avg', 'absent_model_aware_avg']:
        print('Unknown conf_type: {}. Must be "avg", "max" or "box_and_model_avg", or "absent_model_aware_avg"'.format(conf_type))
        sys.exit()

    filtered_boxes = prefilter_boxes(boxes_list, scores_list, labels_list, weights, skip_box_thr)
    if len(filtered_boxes) == 0:
        return np.zeros((0, 4)), np.zeros((0,)), np.zeros((0,)), np.zeros((0, 4))
    
    overall_boxes = []
    original_boxes = []
    for label in filtered_boxes:
        boxes = filtered_boxes[label]
        new_boxes = []
        weighted_boxes = []
        # Clusterize boxes
        for j in range(0, len(boxes)):
            index, best_iou = find_matching_box(weighted_boxes, boxes[j], iou_thr)
            if index != -1:
                new_boxes[index].append(boxes[j])
                weighted_boxes[index] = get_weighted_box(new_boxes[index], conf_type)
            else:
                new_boxes.append([boxes[j].copy()])
                weighted_boxes.append(boxes[j].copy())
        # Rescale confidence based on number of models and boxes
        original_boxes.append(new_boxes)
        for i in range(len(new_boxes)):
            clustered_boxes = np.array(new_boxes[i])
            if conf_type == 'box_and_model_avg':
                # weighted average for boxes
                weighted_boxes[i][1] = weighted_boxes[i][1] * len(clustered_boxes) / weighted_boxes[i][2]
                # identify unique model index by model index column
                _, idx = np.unique(clustered_boxes[:, 3], return_index=True)
                # rescale by unique model weights
                weighted_boxes[i][1] = weighted_boxes[i][1] *  clustered_boxes[idx, 2].sum() / weights.sum()
            elif conf_type == 'absent_model_aware_avg':
                # get unique model index in the cluster
                models = np.unique(clustered_boxes[:, 3]).astype(int)
                # create a mask to get unused model weights
                mask = np.ones(len(weights), dtype=bool)
                mask[models] = False
                # absent model aware weighted average
                weighted_boxes[i][1] = weighted_boxes[i][1] * len(clustered_boxes) / (weighted_boxes[i][2] + weights[mask].sum())
            elif conf_type == 'max':
                weighted_boxes[i][1] = weighted_boxes[i][1] / weights.max()
            elif not allows_overflow:
                weighted_boxes[i][1] = weighted_boxes[i][1] * min(len(weights), len(clustered_boxes)) / weights.sum()
            else:
                weighted_boxes[i][1] = weighted_boxes[i][1] * len(clustered_boxes) / weights.sum()
        overall_boxes.append(np.array(weighted_boxes))
    overall_boxes = np.concatenate(overall_boxes, axis=0)
    sidx = overall_boxes[:, 1].argsort()
    overall_boxes = overall_boxes[sidx[::-1]]
    boxes = overall_boxes[:, 4:]
    scores = overall_boxes[:, 1]
    labels = overall_boxes[:, 0]
    # sort originals accoring to wbf
    original_boxes = original_boxes[0]
    wbfo = [original_boxes[i] for i in sidx[::-1]]
    return boxes, scores, labels, wbfo

Writing wbf_tracking.py


In [8]:
import transforms as T

def get_transform(train):
    transforms = []
    transforms.append(T.PILToTensor())
    transforms.append(T.ConvertImageDtype(torch.float))
    return T.Compose(transforms)

In [9]:
from engine import train_one_epoch, evaluate
import utils

In [10]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [11]:
sys.path.append('/kaggle/input/einops/einops-master')

sys.path.append("../input/pretrained-models-pytorch")
sys.path.append("../input/efficientnet-pytorch")
sys.path.append("/kaggle/input/smp-github/segmentation_models.pytorch-master")

In [12]:
import sys
print(sys.path)
!cp -r /kaggle/input/vitadadapter/hubmap/vitadap/vitadapzip/ ./
sys.path.insert(1, '/kaggle/working/vitadapzip')
print(sys.path)


['/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib', '/kaggle/input/hubmap-hacking-the-human-vasculature', '/opt/conda/lib/python310.zip', '/opt/conda/lib/python3.10', '/opt/conda/lib/python3.10/lib-dynload', '', '/root/.local/lib/python3.10/site-packages', '/opt/conda/lib/python3.10/site-packages', '/src/bq-helper', '/kaggle/input/detection-wheel', '/kaggle/input/einops/einops-master', '../input/pretrained-models-pytorch', '../input/efficientnet-pytorch', '/kaggle/input/smp-github/segmentation_models.pytorch-master']
['/kaggle/working', '/kaggle/working/vitadapzip', '/kaggle/lib/kagglegym', '/kaggle/lib', '/kaggle/input/hubmap-hacking-the-human-vasculature', '/opt/conda/lib/python310.zip', '/opt/conda/lib/python3.10', '/opt/conda/lib/python3.10/lib-dynload', '', '/root/.local/lib/python3.10/site-packages', '/opt/conda/lib/python3.10/site-packages', '/src/bq-helper', '/kaggle/input/detection-wheel', '/kaggle/input/einops/einops-master', '../input/pretrained-models-pytorch', '../

In [13]:

import segmentation_models_pytorch as smp
import mmcv_custom
import mmdet_custom

/kaggle/working/../input/pretrained-models-pytorch/pretrainedmodels/models/dpn.py:255: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if block_type is 'proj':
/kaggle/working/../input/pretrained-models-pytorch/pretrainedmodels/models/dpn.py:258: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif block_type is 'down':
/kaggle/working/../input/pretrained-models-pytorch/pretrainedmodels/models/dpn.py:262: SyntaxWarning: "is" with a literal. Did you mean "=="?
  assert block_type is 'normal'
/opt/conda/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(


In [14]:
import mmdet
from mmdet.apis import init_detector, inference_detector,show_result_pyplot, set_random_seed
from mmdet.models import build_detector
#print(mmdet.__version__)
#print(mmcv.__version__)
#print(mmengine.__version__)

from mmdet.models.backbones import *
# # #check file her

from mmcv import Config

from mmdet.models.backbones.swin import SwinTransformer
from mmdet.models.backbones import cbnet

In [15]:
# import sys
# print(sys.path)
# !cp -r /kaggle/input/cbnetv2-repo ./# 
# sys.path.insert(1, '/kaggle/working/cbnet_repo/')
# print(sys.path)

import mmdet
from mmdet.apis import init_detector, inference_detector,show_result_pyplot, set_random_seed
from mmdet.models import build_detector
#print(mmdet.__version__)
#print(mmcv.__version__)
#print(mmengine.__version__)

from mmdet.models.backbones import *
# # #check file her

from mmcv import Config

from mmdet.models.backbones.swin import SwinTransformer
from mmdet.models.backbones import cbnet

In [16]:
# ===== 融合版：只用 3 個免編譯模型（DetectoRS R50 / ResNeXt101 / CBNetV2）=====
# 移除全部 ViT-Adapter（adap*）→ 不觸發 /ops/ MSDeformAttn 編譯。
from mmcv import Config
from mmdet.apis import init_detector, inference_detector, set_random_seed

configs_path = [
    '/kaggle/input/ds1pretexp1moreaug-htc50-2048-cv408ps/h50psexp1.py',                 # DetectoRS R50
    '/kaggle/input/pretwsiallhtc-resnext101-exp3-augv4-maskloss4/detec101next.py',       # DetectoRS ResNeXt101
    '/kaggle/input/pretexp1cbnetv2-base-2048-basic-exp1-f5cv437/exp1_cbnet_swinb2.py',   # CBNetV2-base
]
ckpt_paths = [
    '/kaggle/input/ds1pretexp1moreaug-htc50-2048-cv408ps/detectors_epoch_23.pth',
    '/kaggle/input/pretwsiallhtc-resnext101-exp3-augv4-maskloss4/best_segm_mAP_epoch_17.pth',
    '/kaggle/input/pretexp1cbnetv2-base-2048-basic-exp1-f5cv437/best_segm_mAP_epoch_21.pth',
]
assert len(configs_path) == len(ckpt_paths) == 3


In [17]:
models = []
for cfg_path, ckpt in zip(configs_path,ckpt_paths):
    cfg = Config.fromfile(cfg_path)
    if 'cbnet' in cfg_path:
        print('small image for cbnet')
        cfg.model.test_cfg.rcnn.score_thr = 0.001

        cfg.model.test_cfg.rcnn.max_per_img = 500

        cfg.model.test_cfg.rcnn.nms.iou_threshold=0.5
        cfg.model.test_cfg.rcnn.mask_thr_binary=0.55
        cfg.data.test.pipeline[1].img_scale= [(2048, 2048)]#
        
    elif 'adap' in cfg_path:
        print('small image for adap')
        cfg.model.test_cfg.rcnn.score_thr = 0.001

        cfg.model.test_cfg.rcnn.max_per_img = 500
        cfg.model.test_cfg.rcnn.nms.type='nms'

        cfg.model.test_cfg.rcnn.nms.iou_threshold=0.5
        cfg.model.test_cfg.rcnn.mask_thr_binary=0.55
        cfg.data.test.pipeline[1].img_scale= [(1600,1600),(1400,1400)]#
        
    else:
        cfg.data.test.pipeline[1].img_scale= [(2048,2048)]#
        cfg.model.test_cfg.rcnn.score_thr = 0.001

        cfg.model.test_cfg.rcnn.max_per_img = 500

        cfg.model.test_cfg.rcnn.nms.iou_threshold=0.5
        cfg.model.test_cfg.rcnn.mask_thr_binary=0.55
    cfg.seed = 69
    set_random_seed(69, deterministic=False)

    print(f'Config:\n{cfg.data.test.pipeline}')
    model = init_detector(cfg, ckpt, device=device)  
    models.append(model)
    del cfg

small image for adap
Config:
[{'type': 'LoadImageFromFile'}, {'type': 'MultiScaleFlipAug', 'img_scale': [(1600, 1600), (1400, 1400)], 'flip': True, 'flip_direction': ['horizontal', 'vertical'], 'transforms': [{'type': 'Resize', 'keep_ratio': True}, {'type': 'RandomFlip'}, {'type': 'Normalize', 'mean': [123.675, 116.28, 103.53], 'std': [58.395, 57.12, 57.375], 'to_rgb': True}, {'type': 'Pad', 'size_divisor': 32}, {'type': 'ImageToTensor', 'keys': ['img']}, {'type': 'Collect', 'keys': ['img']}]}]


/opt/conda/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /usr/local/src/pytorch/aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load checkpoint from local path: /kaggle/input/pretexp4-adapbeitv2lhtc-1400-ds2wsiall-ps60-leak/best_segm_mAP_epoch_18.pth
small image for adap
Config:
[{'type': 'LoadImageFromFile'}, {'type': 'MultiScaleFlipAug', 'img_scale': [(1600, 1600), (1400, 1400)], 'flip': True, 'flip_direction': ['horizontal', 'vertical'], 'transforms': [{'type': 'Resize', 'keep_ratio': True}, {'type': 'RandomFlip'}, {'type': 'Normalize', 'mean': [123.675, 116.28, 103.53], 'std': [58.395, 57.12, 57.375], 'to_rgb': True}, {'type': 'Pad', 'size_divisor': 32}, {'type': 'ImageToTensor', 'keys': ['img']}, {'type': 'Collect', 'keys': ['img']}]}]
load checkpoint from local path: /kaggle/input/pretexp4-adapbeitv2lhtc-1400-ds2wsiall-ps50exp2-lo/best_segm_mAP_epoch_20.pth
Config:
[{'type': 'LoadImageFromFile'}, {'type': 'MultiScaleFlipAug', 'img_scale': [(2048, 2048)], 'flip': True, 'flip_direction': ['horizontal', 'vertical'], 'transforms': [{'type': 'Resize', 'keep_ratio': True}, {'type': 'RandomFlip'}, {'type': 'Norm

In [18]:
all_imgs = glob.glob('/kaggle/input/hubmap-hacking-the-human-vasculature/test/*.tif')
dataset_test = PennFudanDataset(all_imgs, get_transform(train=False))
test_dl = torch.utils.data.DataLoader(
        dataset_test, batch_size=1, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)

In [19]:
# all_imgs

In [20]:
from skimage.morphology import binary_dilation


In [21]:

from skimage.morphology import binary_erosion, binary_dilation, binary_opening, binary_closing

MIN_PIXELS = 40
def nms_predictions(classes, scores, bboxes, masks, 
                    iou_th=.5, shape=(512, 512), weights=[0.5,0.5]):
    he, wd = shape[0], shape[1]
    boxes_list = [[x[0] / wd, x[1] / he, x[2] / wd, x[3] / he]
                  for x in bboxes]
    scores_list = [x for x in scores]
    labels_list = [x for x in classes]
    nms_bboxes, nms_scores, nms_classes = nms(
        boxes=[boxes_list], 
        scores=[scores_list], 
        labels=[labels_list], 
        weights=weights,
        iou_thr=iou_th
    )
    nms_masks = []
    for s in nms_scores:
        nms_masks.append(masks[scores.index(s)])
    nms_scores, nms_classes, nms_masks = zip(*sorted(zip(nms_scores, nms_classes, nms_masks), reverse=True))
    return nms_classes, nms_scores, nms_masks

def ensemble_pred_masks(masks, min_pixels=MIN_PIXELS, shape=(512, 512)):
    result = []
    used = np.zeros(shape, dtype=int) 

    prev_masks = []
    new_masks = []
    for i, mask in enumerate(masks):
        # cont, hier = cv2.findContours(np.array(mask,dtype=np.uint8),cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        # if len(cont)>0:
            # for cnt in cont:
            #     convex_mask = cv2.fillConvexPoly(np.zeros_like(np.array(mask,dtype=np.uint8)),points=cnt, color=1)
            #     fillornot = len(pd.Series((convex_mask==mask).flatten()).value_counts())
            #     if fillornot>1: #fill
            #         mask = convex_mask

        
            # before= mask.sum()
        mask = binary_erosion(binary_dilation(mask))  #post processing 
            # if before!=mask.sum():
               # print('after',mask.sum(),'before',before)
        mask = mask * (1-used)
        if mask.sum() >= 100: # skip predictions with small area
                #     used += mask 
            new_masks.append(mask)
        
    return new_masks



In [22]:
from ensemble_boxes import *

In [23]:
ids = []
heights = []
widths = []
prediction_strings = []

In [24]:
from skimage import measure
from wbf_tracking import *

In [25]:
from ensemble_boxes import *
MODEL_WEIGHTS = [0.25,0.5,0.25]
def bbox_to_key(bbox):
    return str(np.round(bbox, 6))


# Fuse masks that belong to fused boxes
def get_wsf_mask(wbf_box, wbf_org, pmasks, pmasks_lkup, thres=0.5):
    w, h = 512, 512
    mask = np.zeros((w, h), dtype=np.uint8)
    for i in range(len(wbf_org)):
        key = bbox_to_key(wbf_org[i][4:])
        model = int(wbf_org[i][3])
        # try:
        ind = pmasks_lkup[model][key]
        mask = mask + pmasks[model][ind]
        # except:
            # pass
    # convert thres to integer based on number of boxes
    threshold = max(1, int(thres*len(wbf_org)))
    # threshold = 0.0001
            
    # remove pixels outside WBF box
    m2 = np.zeros((w, h), dtype=np.uint8)
    x1 = max(0, int(h * wbf_box[0]))
    y1 = max(0, int(w * wbf_box[1]))
    x2 = min(h, int(h * wbf_box[2]))
    y2 = min(w, int(w * wbf_box[3]))
    # print(x1,x2,y1,y2)
    m2[y1:y2, x1:x2] = 1
    mask = (mask >= threshold) * m2
    return mask.astype(np.uint8)

In [ ]:
# ===== 載入 InternImage 預測（第 4 條 stream）=====
# 來源：mmdet_v3_solution/hubmap_internimage_infer.ipynb 跑出的 internimage_preds.pkl
# 把那個 notebook 的 Output 存成 Kaggle dataset 後，改下面這行 slug。
import pickle
from pycocotools import mask as mask_util

INTERNIMAGE_PKL = '/kaggle/input/internimage-preds/internimage_preds.pkl'  # ← 改成你的 dataset 路徑
W_INTERN = 0.5   # InternImage 相對權重（作者各模 = 1.0）；較弱故給 0.5，之後可調

with open(INTERNIMAGE_PKL, 'rb') as f:
    INTERNIMAGE_PREDS = pickle.load(f)
print(f'InternImage preds: {len(INTERNIMAGE_PREDS)} 圖, '
      f'{sum(len(v) for v in INTERNIMAGE_PREDS.values())} instances, W_INTERN={W_INTERN}')


In [26]:
subm_ids, subm_masks = [], []
sample = None
import mmcv


confidence_thresholds = {0: 0.5, 1: 0.5, 2: 0.8}

for img in all_imgs:
    pred_string = ''

    img_array = mmcv.imread(img,channel_order='rgb')
    [h, w, c] = img_array.shape 
    
    

    
    masks_nms_list = []
    pred_dict_list=  []
    score_nms_list = []
    box_nms_list = []
    class_nms_list = []
    
    
    for modely in models:
        
        pred_dict = {}
        previous_masks = []
        classes_nms = []
        scoresb_nms = []
        bboxesb_nms = []
        weightsb_nms = []     
        
        result = inference_detector(modely,img)

        c = []
        for i, classe in enumerate(result[0]):
            c.append(classe.shape[0])
        
        maxclass = np.argwhere(np.array(c)==np.max(c))[0][0]
        for i, classe in enumerate(result[0]):
            if i==maxclass: #classe.shape != (0, 5):
                bbs = classe
                sgs = result[1][i]
                # print(sgs.shape)
                count = 0

                for bb, sg in zip(bbs,sgs):
                    box = bb[:4]
                    cnf = bb[4]
                    box = [box[0] / 512, box[1] / 512, box[2] / 512, box[3] / 512]
                    
                    if cnf >= 0.00001:
                        mask = np.array(sg,dtype=np.uint8)  
                        previous_masks.append(mask)
                        scoresb_nms.extend([cnf])
                        bboxesb_nms.extend([box])
                        # weights = [weight] * len()
#                         weightsb_nms.extend([weight])
                        pred_dict[bbox_to_key(box)] = count
                        count+= 1
                        classes_nms = [0] * len(previous_masks)
    
            masks_nms_list.append(np.array(previous_masks, dtype=np.uint8))
            score_nms_list.append(np.array(scoresb_nms))
            box_nms_list.append(np.array(bboxesb_nms))

            class_nms_list.append(np.array(classes_nms))
            pred_dict_list.append(pred_dict)
            
            
    # ===== 注入 InternImage 第 4 條 stream（來自 mmdet3.x notebook 的 pkl）=====
    _iid = os.path.basename(img).split('.')[0]
    _ii_masks, _ii_scores, _ii_boxes = [], [], []
    _ii_pd, _cnt = {}, 0
    for _inst in INTERNIMAGE_PREDS.get(_iid, []):
        _m = mask_util.decode(_inst['rle']).astype(np.uint8)       # 512x512
        _x1, _y1, _x2, _y2 = _inst['box']
        _b = [_x1 / 512.0, _y1 / 512.0, _x2 / 512.0, _y2 / 512.0]  # 與作者一致：box/512
        _ii_masks.append(_m); _ii_scores.append(_inst['score']); _ii_boxes.append(_b)
        _ii_pd[bbox_to_key(_b)] = _cnt; _cnt += 1
    _N_author = len(box_nms_list)                                  # 作者模型數（=3）
    masks_nms_list.append(np.array(_ii_masks, dtype=np.uint8))
    score_nms_list.append(np.array(_ii_scores))
    box_nms_list.append(np.array(_ii_boxes))
    class_nms_list.append(np.array([0] * len(_ii_masks)))
    pred_dict_list.append(_ii_pd)
    _wbf_weights = [1.0] * _N_author + [W_INTERN]                  # 作者各 1.0、InternImage W_INTERN

    wbf_boxes, wbf_scores, _, wbf_originals = weighted_boxes_fusion_tracking(box_nms_list, 
                                                                             score_nms_list, 
                                                                             labels_list=class_nms_list, 
#                                                                              weights = [0.20,0.20,0.2,0.15,0.15,0.1],
                                                                             weights = _wbf_weights,
                                                                             
                                                                             iou_thr=0.6, 
                                                                             skip_box_thr=0.01)
    
    
    
    fin_masks = []
    used = np.zeros((512,512), dtype=int)
    for i in range(len(wbf_boxes)):
        
        mask = get_wsf_mask(wbf_boxes[i], wbf_originals[i], masks_nms_list, pred_dict_list, thres=0.2)
        fin_masks.append(mask)
    
    fin_masks = ensemble_pred_masks(fin_masks) 

    m = 0
    for masky, scory in zip(fin_masks, tuple(wbf_scores)):
        masky = masky.astype(bool)       
        
        encoded = encode_binary_mask(masky)
        if m==0:
            pred_string += f"0 {scory} {encoded.decode('utf-8')}"
            m=m+1
        else:
            pred_string += f" 0 {scory} {encoded.decode('utf-8')}"

            
    ids.append(os.path.basename(img).split('.')[0])
    heights.append(h)
    widths.append(w)
            
    prediction_strings.append(pred_string)                    

    

/kaggle/working/vitadapzip/mmdet/datasets/utils.py:66: UserWarning: "ImageToTensor" pipeline is replaced by "DefaultFormatBundle" for batch inference. It is recommended to manually replace it in the test data pipeline in your config file.
  warnings.warn(
/kaggle/working/vitadapzip/mmdet/models/roi_heads/mask_heads/fcn_mask_head.py:245: UserWarning: Scale_factor should be a Tensor or ndarray with shape (4,), float would be deprecated. 
  warn('Scale_factor should be a Tensor or ndarray '
/tmp/ipykernel_23/2079321387.py:11: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  if mask.dtype != np.bool:


In [27]:
# masks_nms

In [28]:
# len(scoresb_nms)

In [29]:
submission = pd.DataFrame()
submission['id'] = ids
submission['height'] = heights
submission['width'] = widths
submission['prediction_string'] = prediction_strings
submission = submission.set_index('id')
submission.to_csv("submission.csv")
submission.head()

,height,width,prediction_string
id,,,
72e40acccadf,512,512,0 0.9782930612564087 eNoLyM8ysUy1t/C08LDwNPIz9...


In [30]:
# submission.loc['72e40acccadf','prediction_string']

In [31]:
!rm -rf mmdetection
!rm -rf packages
# !rm -rf cbnetv2-repo
!rm -rf cbnet_repo
!rm -rf /kaggle/working/vitadapzip

In [32]:
##a